### 실습 1. Chapter 01 전처리 데이터 불러오기

이번 Chapter에서도 Chapter 01에서 만든 다음 파일을 사용합니다.

In [1]:
import pandas as pd

DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())

df_books[["상품명"]].head(10)

데이터 크기: (989, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,"세네카, 오늘을 빼앗기고 있는 당신에게"
1,홍정기의 장수근육 혁명
2,싯다르타
3,머니 트렌드 2027
4,흔한남매 23
5,2026 해커스 투자자산운용사 실전동형모의고사 10회분+리얼 기출족보 3종
6,시대예보: 수고인류의 시간
7,리더는 언제 차이를 만들어내는가
8,판매의 법칙
9,니체의 초월자


### 확인 결과

- CSV 파일이 정상적으로 열렸다.
- 데이터 크기는 **989행, 8개 컬럼**이다.
- `상품명` 컬럼이 존재하며 한글도 정상적으로 출력되었다.
- `상품명`의 결측치는 **0개**이다.

### 실습 2. 분석할 제목 문자열 준비하기

Vectorizer에 전달할 제목을 문자열 형태로 준비합니다.

In [2]:
# 상품명 컬럼의 결측치는 빈 문자열로 바꾸고 문자열로 변환
titles = df_books["상품명"].fillna("").astype(str)

# 상품명 앞뒤의 불필요한 공백 제거
titles = titles.str.strip()

# 내용이 비어 있는 상품명은 제외하고 인덱스 재정리
titles = titles[titles != ""].reset_index(drop=True)

# 실제 분석에 사용할 제목 개수 확인
print("사용할 제목 수:", len(titles))

# 앞의 제목 10개 확인
titles.head(10)

사용할 제목 수: 989


0                        세네카, 오늘을 빼앗기고 있는 당신에게
1                                 홍정기의 장수근육 혁명
2                                         싯다르타
3                                  머니 트렌드 2027
4                                      흔한남매 23
5    2026 해커스 투자자산운용사 실전동형모의고사 10회분+리얼 기출족보 3종
6                               시대예보: 수고인류의 시간
7                            리더는 언제 차이를 만들어내는가
8                                       판매의 법칙
9                                      니체의 초월자
Name: 상품명, dtype: str

### 실행 결과

- 결측치와 앞뒤 공백을 처리하고 빈 제목을 제외했다.
- 분석에 사용할 제목은 총 **989개**이다.
- 제목이 문자열 형태로 정상적으로 준비된 것을 확인했다.

### 실습 3. 머신러닝은 왜 텍스트를 숫자로 바꿀까?

머신러닝 알고리즘은 일반적으로 숫자를 입력받아 계산합니다.

예를 들어 다음과 같은 숫자는 계산할 수 있습니다.

도서 제목

        ↓

사용된 단어 확인

        ↓

단어마다 열(column) 생성

        ↓

등장 여부 또는 등장 횟수를 숫자로 기록

        ↓

숫자 벡터

### 실습 4. 아주 작은 예제로 먼저 이해하기

실제 1,000권 정도의 도서 제목을 바로 사용하면 행과 열이 많아져 원리를 보기 어렵습니다.

먼저 세 문장만 사용합니다.

In [3]:
# 원리를 이해하기 위한 작은 문장 3개 준비
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

# 문장에 등장하는 단어 순서 지정
words = ["데이터", "머신러닝", "분석", "입문", "파이썬"]

# 첫 번째 문장에서 각 단어가 등장한 횟수 계산
first_vector = []

for word in words:
    count = sample_docs[0].split().count(word)
    first_vector.append(count)

print("첫 번째 문장:", sample_docs[0])
print("단어 순서:", words)
print("숫자 벡터:", first_vector)

첫 번째 문장: 파이썬 데이터 분석
단어 순서: ['데이터', '머신러닝', '분석', '입문', '파이썬']
숫자 벡터: [1, 0, 1, 0, 1]


### 실행 결과

첫 번째 문장 `파이썬 데이터 분석`을 단어별 등장 횟수로 변환한 결과는 `[1, 0, 1, 0, 1]`이다. 각 숫자는 의미 점수가 아니라 지정한 단어가 문장에 등장한 횟수를 나타낸다.

### 실습 5. Bag of Words 이해하기

앞의 방식은 단어를 하나의 주머니에 넣고 몇 번 등장했는지 세는 것과 비슷합니다.

이 개념을 Bag of Words, 줄여서 BoW라고 부릅니다.

### Bag of Words 이해

Bag of Words는 단어의 순서보다 어떤 단어가 몇 번 등장했는지에 초점을 둔다.

`파이썬 데이터 분석`과 `데이터 파이썬 분석`은 단어 순서는 다르지만 같은 단어가 같은 횟수로 등장하므로 동일한 Count 벡터가 될 수 있다.

따라서 Bag of Words는 단순하고 이해하기 쉽지만 단어의 순서와 문맥을 충분히 표현하지 못한다.

### 실습 6. CountVectorizer 설치 확인하기

CountVectorizer는 scikit-learn에 포함되어 있습니다.

설치가 필요한 경우 다음과 같이 실행합니다.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

### 실습 7. 작은 예제에 CountVectorizer 적용하기

In [5]:
# CountVectorizer 불러오기
from sklearn.feature_extraction.text import CountVectorizer

# 실습에 사용할 작은 문장 3개
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

# 단어 등장 횟수를 계산할 Vectorizer 생성
count_vectorizer = CountVectorizer()

# 단어 사전을 학습하고 문장을 숫자 벡터로 변환
X_count_sample = count_vectorizer.fit_transform(sample_docs)

# 변환 결과의 크기 확인
print("변환 완료")
print("행렬 크기:", X_count_sample.shape)

변환 완료
행렬 크기: (3, 5)


### 실행 결과

CountVectorizer를 이용해 3개의 문장을 숫자 벡터로 변환했다.

행렬 크기는 `(3, 5)`로, **3개의 문장과 5개의 단어**로 구성된 것을 확인했다. `fit_transform()`은 단어 사전을 학습하고 각 문장을 숫자로 변환하는 작업을 함께 수행한다.

### 실습 8. 생성된 단어 사전 확인하기

Vectorizer가 어떤 단어를 열로 만들었는지 확인합니다.

In [6]:
# CountVectorizer가 만든 단어 목록 가져오기
feature_names = count_vectorizer.get_feature_names_out()

# 단어 목록 확인
print("생성된 단어 사전:", feature_names)

생성된 단어 사전: ['데이터' '머신러닝' '분석' '입문' '파이썬']


### 실행 결과

CountVectorizer가 만든 단어 사전은 `데이터`, `머신러닝`, `분석`, `입문`, `파이썬`으로 확인되었다.

숫자 벡터의 각 값은 이 단어 순서를 기준으로 해석해야 한다.

### 실습 9. 단어-문서 행렬 확인하기

작은 예제이므로 행렬 전체를 배열로 바꿔 확인해 봅니다.

In [7]:
# 표를 만들기 위해 pandas 불러오기
import pandas as pd

# 희소 행렬을 일반 배열로 변환하고 표로 만들기
sample_count_df = pd.DataFrame(
    X_count_sample.toarray(),
    columns=feature_names,  # 열: 단어
    index=sample_docs,      # 행: 문장
)

# 단어-문서 행렬 확인
sample_count_df

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


### 실행 결과

단어-문서 행렬에서 **행은 문장**, **열은 단어**, **값은 단어가 등장한 횟수**를 의미한다.

첫 번째 문장 `파이썬 데이터 분석`에서는 `데이터`, `분석`, `파이썬`이 각각 1이고, `머신러닝`과 `입문`은 0으로 나타났다.

### 실습 10. 같은 단어가 여러 번 나오면 어떻게 될까?

다음 문장을 추가해 봅니다.

In [8]:
# 같은 단어가 반복되는 문장 준비
repeat_docs = [
    "파이썬 파이썬 데이터",
    "데이터 분석",
]

# 새로운 CountVectorizer 생성
repeat_vectorizer = CountVectorizer()

# 단어 사전을 학습하고 문장을 숫자로 변환
X_repeat = repeat_vectorizer.fit_transform(repeat_docs)

# 결과를 표로 만들기
repeat_df = pd.DataFrame(
    X_repeat.toarray(),
    columns=repeat_vectorizer.get_feature_names_out(),
    index=repeat_docs,
)

# 결과 확인
repeat_df

,데이터,분석,파이썬
파이썬 파이썬 데이터,1,0,2
데이터 분석,1,1,0


### 실행 결과

첫 번째 문장 `파이썬 파이썬 데이터`에서 `파이썬`이 두 번 등장하여 값이 **2**로 나타났다.

따라서 CountVectorizer는 단어의 존재 여부가 아니라 **각 단어의 등장 횟수**를 기록한다.

### 실습 11. CountVectorizer의 기본 토큰 기준 확인하기

CountVectorizer()는 기본 설정에서 문자열을 내부적으로 나누어 토큰을 만듭니다.

기본 토큰 패턴은 일반적으로 두 글자 이상의 단어 문자를 대상으로 합니다.

따라서 한 글자 토큰은 기본 설정에서 제외될 수 있습니다.

예를 들어 다음 결과를 직접 확인해 봅니다.

In [9]:
# 한 글자와 두 글자 이상의 단어가 포함된 문장
test_docs = [
    "AI 데이터 분석 R 파이썬",
]

# 기본 설정의 CountVectorizer 생성
test_vectorizer = CountVectorizer()

# 단어를 학습하고 숫자 벡터로 변환
X_test = test_vectorizer.fit_transform(test_docs)

# 만들어진 단어 목록 확인
print("생성된 단어:", test_vectorizer.get_feature_names_out())

생성된 단어: ['ai' '데이터' '분석' '파이썬']


### 실행 결과

생성된 단어는 `ai`, `데이터`, `분석`, `파이썬`으로 확인되었다.

한 글자인 `R`은 기본 토큰 기준에서 제외되었고, 두 글자인 `AI`는 소문자 `ai`로 변환되어 포함되었다.

### CountVectorizer의 기본 토큰 기준

CountVectorizer는 기본적으로 **두 글자 이상의 단어**를 토큰으로 사용한다.

실행 결과 `AI`, `데이터`, `분석`, `파이썬`은 포함되었지만, 한 글자인 `R`은 제외되었다. 또한 영어 대문자 `AI`는 기본 설정에 따라 소문자 `ai`로 변환되었다.

따라서 분석 결과에 한 글자 단어가 보이지 않거나 영문이 소문자로 나타나는 것은 오류가 아니라 CountVectorizer의 기본 설정 때문이다. 분석 목적에 따라 토큰 기준을 따로 설정해야 한다.

### 실습 12. 실제 도서 제목에 CountVectorizer 적용하기

이제 실제 상품명 데이터에 적용합니다.

In [10]:
# 실제 도서 제목을 분석할 CountVectorizer 생성
count_vectorizer = CountVectorizer()

# 도서 제목의 단어 사전을 학습하고 숫자 벡터로 변환
X_count = count_vectorizer.fit_transform(titles)

# 생성된 단어 목록 가져오기
count_terms = count_vectorizer.get_feature_names_out()

# 행렬의 크기 확인
print("문서 수:", X_count.shape[0])
print("단어 수:", X_count.shape[1])
print("행렬 크기:", X_count.shape)

# 제목 수와 행렬의 문서 수가 같은지 확인
print("제목 수와 문서 수가 같은가?:", len(titles) == X_count.shape[0])

문서 수: 989
단어 수: 2192
행렬 크기: (989, 2192)
제목 수와 문서 수가 같은가?: True


### 실행 결과

총 **989개의 도서 제목**에서 **2,192개의 단어**가 생성되었다.

Count 행렬의 크기는 `(989, 2192)`이며, 행은 도서 제목이고 열은 생성된 단어를 의미한다. 제목 수와 행렬의 문서 수가 일치하여 정상적으로 변환된 것을 확인했다.

### 실습 13. vocabulary_ 확인하기

Vectorizer 내부에는 단어와 열 번호의 대응 정보가 있습니다.

In [11]:
# 단어와 열 번호의 연결 정보 확인
vocabulary_items = list(count_vectorizer.vocabulary_.items())

# 앞의 20개 확인
vocabulary_items[:20]

[('세네카', 1160),
 ('오늘을', 1477),
 ('빼앗기고', 1052),
 ('있는', 1664),
 ('당신에게', 647),
 ('홍정기의', 2159),
 ('장수근육', 1706),
 ('혁명', 2138),
 ('싯다르타', 1313),
 ('머니', 834),
 ('트렌드', 1965),
 ('2027', 63),
 ('흔한남매', 2186),
 ('23', 74),
 ('2026', 61),
 ('해커스', 2097),
 ('투자자산운용사', 1963),
 ('실전동형모의고사', 1293),
 ('10회분', 17),
 ('리얼', 769)]

### 확인할 핵심

`vocabulary_`는 각 단어와 Count 행렬의 열 번호를 연결한 정보이다.

`('파이썬', 123)`에서 `123`은 파이썬이 123번 등장했다는 뜻이 아니라, `파이썬`이 행렬의 123번 열에 있다는 뜻이다.

### 실행 결과

`vocabulary_`를 확인한 결과 각 단어와 열 번호가 함께 출력되었다.

예를 들어 `세네카`의 `1160`은 등장 횟수가 아니라 Count 행렬에서 해당 단어가 위치한 **열 번호**를 의미한다. 단어의 등장 횟수는 행렬의 실제 셀 값에서 확인해야 한다.

### 실습 14. 희소 행렬(Sparse Matrix) 이해하기

실제 도서 제목 전체를 벡터화하면 많은 값이 0이 됩니다.

예를 들어 특정 도서 제목에 파이썬이라는 단어가 없다면 그 열의 값은 0입니다.

수백 또는 수천 개 단어를 만들면 대부분의 문서에서 대부분의 단어는 등장하지 않습니다.

In [12]:
# CountVectorizer가 반환한 행렬의 자료형 확인
print("행렬 타입:", type(X_count))

# 행렬 크기 확인
print("행렬 크기:", X_count.shape)

# 실제로 저장된 0이 아닌 값의 개수 확인
print("0이 아닌 값의 개수:", X_count.nnz)

행렬 타입: <class 'scipy.sparse._csr.csr_matrix'>
행렬 크기: (989, 2192)
0이 아닌 값의 개수: 3921


### 희소 행렬 이해

CountVectorizer는 대부분의 값이 0인 텍스트 데이터를 효율적으로 저장하기 위해 희소 행렬을 반환한다.

전체 도서 제목 행렬을 일반 배열로 변환하면 메모리를 많이 사용할 수 있으므로, 필요한 행이나 집계 결과만 확인하는 것이 안전하다.

### 실행 결과

Count 행렬은 `csr_matrix` 형태의 **희소 행렬**이며, 크기는 `(989, 2192)`이다.

전체 행렬 중 실제로 0이 아닌 값은 **3,921개**만 존재한다. 대부분의 값이 0이므로 전체를 일반 배열로 변환하지 않고 희소 행렬 형태로 사용하는 것이 효율적이다.

### 실습 15. 실제 데이터의 첫 번째 제목 벡터 확인하기

첫 번째 도서 제목을 확인합니다.

In [13]:
# 첫 번째 도서 제목 확인
print("첫 번째 제목:", titles.iloc[0])

# 첫 번째 제목에 해당하는 Count 벡터 가져오기
first_row = X_count.getrow(0)

# 값이 0이 아닌 단어의 열 번호와 등장 횟수
indices = first_row.indices
values = first_row.data

# 열 번호를 실제 단어로 변환
first_title_terms = [
    (count_terms[index], value)
    for index, value in zip(indices, values)
]

# 첫 번째 제목에서 추출된 단어와 등장 횟수 확인
print("추출 결과:", first_title_terms)

첫 번째 제목: 세네카, 오늘을 빼앗기고 있는 당신에게
추출 결과: [('세네카', np.int64(1)), ('오늘을', np.int64(1)), ('빼앗기고', np.int64(1)), ('있는', np.int64(1)), ('당신에게', np.int64(1))]


### 실행 결과

첫 번째 제목 `세네카, 오늘을 빼앗기고 있는 당신에게`에서 `세네카`, `오늘을`, `빼앗기고`, `있는`, `당신에게`가 추출되었다.

모든 단어가 원래 제목에 존재하며 각각 한 번씩 등장하여 값이 모두 **1**로 나타났다. 쉼표는 단어로 포함되지 않았다.

### 실습 16. 전체 데이터에서 많이 등장한 단어 확인하기

Count 행렬의 각 열을 합하면 전체 문서에서 각 단어가 등장한 총 횟수를 계산할 수 있습니다.

In [14]:
# 배열 계산을 위해 numpy 불러오기
import numpy as np

# Count 행렬의 각 단어 열을 세로로 합산
count_sums = np.asarray(X_count.sum(axis=0)).ravel()

# 단어와 전체 등장 횟수를 표로 만들기
count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums,
})

# 등장 횟수가 많은 순서로 정렬
count_summary = count_summary.sort_values(
    "전체등장횟수",
    ascending=False,
).reset_index(drop=True)

# 상위 30개 단어 확인
count_summary.head(30)

,단어,전체등장횟수
0,2027,93
1,2026,57
2,해커스,38
3,기본서,30
4,세트,25
5,에디션,22
6,기출문제집,21
7,토익,20
8,the,18
9,기념,17


### 실행 결과

전체 도서 제목에서 가장 많이 등장한 단어는 `2027`로 **93회**, 다음은 `2026`으로 **57회** 나타났다.

그 외에 `해커스` 38회, `기본서` 30회, `세트` 25회 등이 상위에 포함되었다. 숫자와 영문도 CountVectorizer의 토큰으로 함께 추출된 것을 확인했다.

이번 결과는 형태소 분석과 불용어 처리를 적용한 Chapter 02와 토큰화 조건이 다르므로 단어 순위에 차이가 있을 수 있다.

### 실습 17. Count 상위 단어 저장하기

상위 30개 단어를 저장합니다.

In [15]:
# 전체 등장 횟수가 높은 상위 30개 단어 선택
count_top30 = count_summary.head(30)

# CSV 파일로 저장
count_top30.to_csv(
    "chapter03_count_top_terms.csv",
    index=False,
    encoding="utf-8-sig",
)

# 저장한 CSV 파일을 다시 불러와 앞의 5개 확인
saved_count_top30 = pd.read_csv(
    "chapter03_count_top_terms.csv",
    encoding="utf-8-sig",
)

print("저장된 행 수:", len(saved_count_top30))
saved_count_top30.head()

저장된 행 수: 30


,단어,전체등장횟수
0,2027,93
1,2026,57
2,해커스,38
3,기본서,30
4,세트,25


### 실행 결과

Count 기준 상위 단어 30개를 `chapter03_count_top_terms.csv` 파일로 저장했다.

파일을 다시 불러온 결과 **30행**이 정상적으로 저장되었으며, 상위 단어는 `2027`, `2026`, `해커스`, `기본서`, `세트` 순으로 확인되었다.

### 실습 18. Count 방식의 한계 생각해 보기

CountVectorizer는 이해하기 쉽지만 한 가지 중요한 문제가 있습니다.

어떤 단어가 거의 모든 문서에서 반복해서 등장한다고 가정합니다.

### CountVectorizer의 한계

CountVectorizer는 단어의 전체 등장 횟수를 쉽게 확인할 수 있지만, 자주 등장한다고 해서 반드시 중요한 단어인 것은 아니다.

예를 들어 여러 제목에 공통으로 등장하는 단어는 횟수가 높아도 각 도서의 특징을 구분하는 데는 도움이 적을 수 있다. 반대로 적은 문서에서만 등장하는 단어가 특정 도서의 특징을 더 잘 나타낼 수도 있다.

이러한 Count 방식의 한계를 보완하기 위해 전체 문서에서의 흔함과 희소성을 함께 고려하는 TF-IDF를 사용한다.

### 실습 19. TF 이해하기

TF는 Term Frequency의 약자입니다.

초보자 단계에서는 다음처럼 이해하면 충분합니다.

### TF 이해

TF(Term Frequency)는 **한 문서 안에서 특정 단어가 얼마나 자주 등장하는지**를 나타낸다.

예를 들어 `파이썬 파이썬 데이터 분석`이라는 문장에서 `파이썬`은 2번, `데이터`와 `분석`은 각각 1번 등장한다. 따라서 이 문서 안에서는 `파이썬`의 빈도가 더 높다.

다만 최종 TF-IDF 값에는 정규화와 IDF도 함께 적용되므로, TF-IDF 값을 단순한 등장 횟수로 해석하면 안 된다.

### 실습 20. DF 이해하기

DF는 Document Frequency입니다.

### DF 이해

DF(Document Frequency)는 **특정 단어가 몇 개의 문서에 등장하는지**를 나타낸다.

예를 들어 100개의 문서 중 `데이터`가 80개 문서에 등장하면 DF는 80이다. 단어가 한 문서에서 여러 번 나와도 등장한 문서는 1개로 계산한다.

DF가 높으면 여러 문서에 널리 등장하는 단어이고, DF가 낮으면 일부 문서에만 등장하는 드문 단어라는 뜻이다.

### 실습 21. IDF 이해하기

IDF는 Inverse Document Frequency입니다.

핵심 생각은 다음과 같습니다.

### IDF 이해

IDF(Inverse Document Frequency)는 **전체 문서에서 특정 단어가 얼마나 흔하거나 드문지**를 나타낸다.

거의 모든 문서에 등장하는 흔한 단어는 문서를 구분하는 힘이 작기 때문에 IDF가 상대적으로 낮아진다. 반대로 일부 문서에만 등장하는 드문 단어는 문서를 구분하는 데 도움이 될 수 있어 IDF가 상대적으로 높아진다.

따라서 IDF는 흔한 단어의 영향은 낮추고, 드문 단어의 영향은 높이는 역할을 한다.

### 실습 22. TF-IDF를 한 문장으로 정리하기

TF-IDF는 다음 두 관점을 결합합니다.

### TF-IDF 이해

TF-IDF는 **한 문서에서는 자주 등장하지만 전체 문서에서는 흔하지 않은 단어에 상대적으로 높은 가중치를 부여하는 방법**이다.

- TF: 현재 문서에서 해당 단어가 얼마나 자주 등장하는가?
- IDF: 해당 단어가 전체 문서에서 얼마나 드문가?

TF-IDF 값이 높다는 것은 현실에서 절대적으로 중요한 단어라는 뜻이 아니라, 현재 문서 집합 안에서 해당 문서의 특징을 상대적으로 잘 나타낼 가능성이 있다는 뜻이다.

### 실습 23. 작은 예제로 TfidfVectorizer 적용하기

In [16]:
# TfidfVectorizer 불러오기
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorizer 생성
tfidf_sample_vectorizer = TfidfVectorizer()

# 작은 예제 문장을 TF-IDF 벡터로 변환
X_tfidf_sample = tfidf_sample_vectorizer.fit_transform(sample_docs)

# 생성된 단어 목록 가져오기
tfidf_sample_terms = tfidf_sample_vectorizer.get_feature_names_out()

# TF-IDF 결과를 표로 만들기
tfidf_sample_df = pd.DataFrame(
    X_tfidf_sample.toarray(),
    columns=tfidf_sample_terms,
    index=sample_docs,
)

# 소수점 셋째 자리까지 표시
tfidf_sample_df.round(3)

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,0.577,0.000,0.577,0.000,0.577
파이썬 머신러닝,0.000,0.796,0.000,0.000,0.605
데이터 분석 입문,0.518,0.000,0.518,0.681,0.000


### 실행 결과

TF-IDF 결과는 단어 등장 횟수가 아닌 가중치이므로 소수로 나타났다.

한 문서에만 등장한 `머신러닝`은 **0.796**, `입문`은 **0.681**로 비교적 높은 값을 가졌다. 반면 여러 문서에 반복해서 등장한 `파이썬`, `데이터`, `분석`은 상대적으로 낮은 값을 보였다.

이를 통해 TF-IDF는 현재 문서에서의 빈도뿐 아니라 전체 문서에서 단어가 얼마나 흔한지도 함께 반영한다는 것을 확인했다.

### 실습 24. TfidfVectorizer가 만든 단어 확인하기

In [17]:
# TfidfVectorizer가 만든 단어 목록 확인
print(
    "생성된 단어:",
    tfidf_sample_vectorizer.get_feature_names_out(),
)

생성된 단어: ['데이터' '머신러닝' '분석' '입문' '파이썬']


### 실행 결과

TfidfVectorizer가 만든 단어는 `데이터`, `머신러닝`, `분석`, `입문`, `파이썬`으로 확인되었다.

TF-IDF 행렬에서도 행은 문서, 열은 단어를 의미한다. 다만 각 셀의 값은 단어 등장 횟수가 아니라 해당 단어의 TF-IDF 가중치이다.

### 실습 25. 단어별 IDF 값 확인하기

TfidfVectorizer가 학습한 IDF 값을 직접 확인할 수 있습니다.

In [18]:
# 단어와 IDF 값을 표로 만들기
idf_df = pd.DataFrame({
    "단어": tfidf_sample_vectorizer.get_feature_names_out(),
    "IDF": tfidf_sample_vectorizer.idf_,
})

# IDF가 높은 순서로 정렬
idf_df = idf_df.sort_values(
    "IDF",
    ascending=False,
).reset_index(drop=True)

# 결과 확인
idf_df

,단어,IDF
0,머신러닝,1.693147
1,입문,1.693147
2,데이터,1.287682
3,분석,1.287682
4,파이썬,1.287682


### 실행 결과

한 문서에만 등장한 `머신러닝`과 `입문`의 IDF는 **1.693147**로 가장 높았다.

두 문서에 등장한 `데이터`, `분석`, `파이썬`의 IDF는 **1.287682**로 상대적으로 낮았다.

이를 통해 전체 문서에서 드물게 등장하는 단어일수록 IDF가 높아지고, 여러 문서에 등장하는 단어일수록 IDF가 낮아지는 것을 확인했다.

### 실습 26. 실제 도서 제목을 TF-IDF로 변환하기

이제 실제 데이터에 적용합니다.

In [19]:
# 실제 도서 제목을 분석할 TF-IDF Vectorizer 생성
tfidf_vectorizer = TfidfVectorizer()

# 도서 제목을 TF-IDF 벡터로 변환
X_tfidf = tfidf_vectorizer.fit_transform(titles)

# 생성된 단어 목록 가져오기
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

# TF-IDF 행렬 크기 확인
print("TF-IDF 행렬 크기:", X_tfidf.shape)
print("문서 수:", X_tfidf.shape[0])
print("단어 수:", X_tfidf.shape[1])

# Count 행렬과 크기 비교
print("Count shape:", X_count.shape)
print("TF-IDF shape:", X_tfidf.shape)
print("두 행렬의 크기가 같은가?:", X_count.shape == X_tfidf.shape)

TF-IDF 행렬 크기: (989, 2192)
문서 수: 989
단어 수: 2192
Count shape: (989, 2192)
TF-IDF shape: (989, 2192)
두 행렬의 크기가 같은가?: True


### 실행 결과

실제 도서 제목을 TF-IDF로 변환한 결과 행렬 크기는 `(989, 2192)`로 확인되었다.

Count 행렬과 TF-IDF 행렬은 모두 **989개의 문서와 2,192개의 단어**로 구성되어 크기가 같았다. 두 행렬의 구조는 같지만, Count는 등장 횟수를 저장하고 TF-IDF는 단어의 상대적 가중치를 저장한다.

### 실습 27. 첫 번째 도서의 TF-IDF 주요 단어 확인하기

첫 번째 제목을 다시 확인합니다.

In [20]:
# 첫 번째 도서 제목 확인
print("도서 제목:", titles.iloc[0])

# 첫 번째 도서의 TF-IDF 벡터 가져오기
first_tfidf_row = X_tfidf.getrow(0)

# 값이 0보다 큰 단어와 TF-IDF 값을 표로 만들기
first_tfidf_df = pd.DataFrame({
    "단어": tfidf_terms[first_tfidf_row.indices],
    "TF-IDF": first_tfidf_row.data,
})

# TF-IDF가 높은 순서로 정렬
first_tfidf_df = first_tfidf_df.sort_values(
    "TF-IDF",
    ascending=False,
).reset_index(drop=True)

# 소수점 여섯째 자리까지 확인
first_tfidf_df.round(6)

도서 제목: 세네카, 오늘을 빼앗기고 있는 당신에게


,단어,TF-IDF
0,세네카,0.458262
1,오늘을,0.458262
2,빼앗기고,0.458262
3,당신에게,0.458262
4,있는,0.399979


### 실행 결과

첫 번째 도서 제목에서 `세네카`, `오늘을`, `빼앗기고`, `당신에게`의 TF-IDF는 각각 **0.458262**로 가장 높았다.

`있는`은 **0.399979**로 상대적으로 낮게 나타났다. 이는 `있는`이 다른 도서 제목에도 등장하여 전체 문서에서 상대적으로 더 흔한 단어이기 때문이다.

출력된 모든 단어가 원래 제목에 존재하는 것을 확인했다.

### 실습 28. 여러 도서의 주요 TF-IDF 단어 확인 함수 만들기

반복해서 확인하기 위해 작은 함수를 만들 수 있습니다.

In [21]:
# 원하는 도서의 주요 TF-IDF 단어를 확인하는 함수
def show_top_tfidf_terms(doc_index, top_n=5):
    # 선택한 도서의 TF-IDF 벡터 가져오기
    row = X_tfidf.getrow(doc_index)

    # 단어와 TF-IDF 값을 표로 만들기
    result = pd.DataFrame({
        "단어": tfidf_terms[row.indices],
        "TF-IDF": row.data,
    })

    # TF-IDF가 높은 단어를 원하는 개수만큼 선택
    result = result.sort_values(
        "TF-IDF",
        ascending=False,
    ).head(top_n)

    # 도서 제목 출력
    print("도서 제목:", titles.iloc[doc_index])

    return result.reset_index(drop=True).round(6)


# 여러 도서의 주요 단어 확인
display(show_top_tfidf_terms(0, top_n=5))
display(show_top_tfidf_terms(10, top_n=5))
display(show_top_tfidf_terms(100, top_n=5))

도서 제목: 세네카, 오늘을 빼앗기고 있는 당신에게


,단어,TF-IDF
0,세네카,0.458262
1,오늘을,0.458262
2,빼앗기고,0.458262
3,당신에게,0.458262
4,있는,0.399979


도서 제목: 군주론 : 쇼츠 에디션


,단어,TF-IDF
0,쇼츠,0.655469
1,군주론,0.618580
2,에디션,0.433265


도서 제목: 일 마키아벨리


,단어,TF-IDF
0,마키아벨리,1.0


### 실행 결과

여러 도서의 주요 TF-IDF 단어를 함수로 확인했다.

- `세네카, 오늘을 빼앗기고 있는 당신에게`에서는 `세네카`, `오늘을`, `빼앗기고`, `당신에게`가 높은 값을 보였다.
- `군주론 : 쇼츠 에디션`에서는 `쇼츠`가 가장 높고, `군주론`, `에디션` 순으로 나타났다.
- `일 마키아벨리`에서는 한 글자인 `일`이 제외되어 `마키아벨리`만 남았고 TF-IDF 값은 **1.0**으로 나타났다.

이를 통해 도서마다 현재 문서 집합에서 상대적으로 특징적인 단어가 다르게 나타나는 것을 확인했다.

### 실습 29. 전체 데이터에서 평균 TF-IDF가 높은 단어 확인하기

각 단어의 TF-IDF 값을 전체 문서에서 평균내어 탐색할 수 있습니다.

In [22]:
# 단어별 TF-IDF 값을 전체 문서에서 평균 계산
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

# 단어와 평균 TF-IDF를 표로 만들기
tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "평균_TFIDF": mean_tfidf,
})

# 평균 TF-IDF가 높은 순서로 정렬
tfidf_summary = tfidf_summary.sort_values(
    "평균_TFIDF",
    ascending=False,
).reset_index(drop=True)

# 상위 30개 확인
tfidf_summary.head(30)

,단어,평균_TFIDF
0,2027,0.022807
1,2026,0.014421
2,해커스,0.010282
3,기본서,0.009226
4,세트,0.008907
5,에디션,0.007372
6,기출문제집,0.007274
7,토익,0.006968
8,the,0.006315
9,해커스경찰,0.005791


### 실행 결과

전체 문서의 평균 TF-IDF를 계산한 결과 `2027`이 **0.022807**로 가장 높았고, `2026`, `해커스`, `기본서`, `세트` 순으로 나타났다.

이 결과는 해당 단어들이 현재 도서 제목 데이터에서 상대적으로 큰 텍스트 특징을 가진다는 의미이다. 평균 TF-IDF가 높다고 해서 도서 판매의 원인이나 현실에서 절대적으로 중요한 단어라고 해석할 수는 없다.

### 실습 30. TF-IDF 상위 단어 저장하기

In [23]:
# 평균 TF-IDF 상위 30개 선택
tfidf_top30 = tfidf_summary.head(30)

# CSV 파일로 저장
tfidf_top30.to_csv(
    "chapter03_tfidf_top_terms.csv",
    index=False,
    encoding="utf-8-sig",
)

# 저장한 파일을 다시 불러와 확인
saved_tfidf_top30 = pd.read_csv(
    "chapter03_tfidf_top_terms.csv",
    encoding="utf-8-sig",
)

print("저장된 행 수:", len(saved_tfidf_top30))
saved_tfidf_top30.head()

저장된 행 수: 30


,단어,평균_TFIDF
0,2027,0.022807
1,2026,0.014421
2,해커스,0.010282
3,기본서,0.009226
4,세트,0.008907


### 실행 결과

평균 TF-IDF 기준 상위 단어 30개를 `chapter03_tfidf_top_terms.csv` 파일로 저장했다.

파일을 다시 불러온 결과 **30행**이 정상적으로 저장되었으며, 상위 단어는 `2027`, `2026`, `해커스`, `기본서`, `세트` 순으로 확인되었다.

### 실습 31. Count 상위 단어와 TF-IDF 상위 단어 비교하기

두 결과를 나란히 비교합니다.

In [24]:
# Count 상위 단어와 평균 TF-IDF 상위 단어를 나란히 비교
comparison = pd.DataFrame({
    "Count_상위단어": count_top30["단어"].reset_index(drop=True),
    "Count_전체등장횟수": count_top30["전체등장횟수"].reset_index(drop=True),
    "TFIDF_상위단어": tfidf_top30["단어"].reset_index(drop=True),
    "평균_TFIDF": tfidf_top30["평균_TFIDF"].reset_index(drop=True),
})

# 상위 20개 비교
comparison.head(20)

,Count_상위단어,Count_전체등장횟수,TFIDF_상위단어,평균_TFIDF
0,2027,93,2027,0.022807
1,2026,57,2026,0.014421
2,해커스,38,해커스,0.010282
3,기본서,30,기본서,0.009226
4,세트,25,세트,0.008907
5,에디션,22,에디션,0.007372
6,기출문제집,21,기출문제집,0.007274
7,토익,20,토익,0.006968
8,the,18,the,0.006315
9,기념,17,해커스경찰,0.005791


### 실행 결과

Count와 평균 TF-IDF를 비교한 결과 상위 9개 단어는 `2027`, `2026`, `해커스`, `기본서`, `세트`, `에디션`, `기출문제집`, `토익`, `the`로 순위가 같았다.

이후에는 순위 차이가 나타났다. 예를 들어 `해커스경찰`, `공단기`, `읽는`은 TF-IDF에서 상대적으로 순위가 높아졌고, `끝내는`, `모의고사`, `ncs` 등은 낮아졌다.

이를 통해 단순 등장 횟수와 전체 문서에서의 흔함을 반영한 TF-IDF 가중치는 서로 다를 수 있음을 확인했다.

### 실습 32. 특정 단어가 몇 개 문서에 등장하는지 확인하기

TF-IDF 차이를 이해하려면 특정 단어의 문서 빈도를 직접 확인해 보는 것이 좋습니다.

In [25]:
# 특정 단어가 등장한 문서 수를 구하는 함수
def document_frequency(term):
    # 단어 사전에 없는 단어라면 0 반환
    if term not in count_vectorizer.vocabulary_:
        return 0

    # 해당 단어가 위치한 열 번호 가져오기
    column_index = count_vectorizer.vocabulary_[term]

    # 전체 문서에서 해당 단어의 열만 선택
    column = X_count[:, column_index]

    # 값이 0보다 큰 문서의 개수 반환
    return int((column > 0).sum())


# 실제 단어의 문서 빈도 확인
print("2027 DF:", document_frequency("2027"))
print("해커스 DF:", document_frequency("해커스"))
print("데이터 DF:", document_frequency("데이터"))
print("파이썬 DF:", document_frequency("파이썬"))

2027 DF: 93
해커스 DF: 38
데이터 DF: 0
파이썬 DF: 0


### 실행 결과

`2027`은 **93개 문서**, `해커스`는 **38개 문서**에 등장했다.

`데이터`와 `파이썬`은 실제 도서 제목의 단어 사전에 존재하지 않아 DF가 **0**으로 나타났다.

DF는 단어의 전체 등장 횟수가 아니라 해당 단어가 등장한 문서의 개수를 의미한다.

### 실습 33. CountVectorizer와 TfidfVectorizer의 역할 비교

두 Vectorizer를 다음처럼 정리할 수 있습니다.

### CountVectorizer와 TfidfVectorizer 비교

| 구분 | CountVectorizer | TfidfVectorizer |
| --- | --- | --- |
| 기본 값 | 단어 등장 횟수 | TF-IDF 가중치 |
| 값 형태 | 주로 정수 | 실수 |
| 전체 문서의 흔함 고려 | 고려하지 않음 | IDF로 고려 |
| 장점 | 단순하고 직관적 | 흔한 단어의 영향을 조정함 |
| 주요 활용 | 빈도 확인 | 분류·유사도 등 텍스트 특징 |

CountVectorizer는 단어가 몇 번 등장했는지 보여주고, TfidfVectorizer는 한 문서에서의 빈도와 전체 문서에서의 희소성을 함께 반영한다.

둘 중 하나가 항상 더 좋은 것은 아니며 분석 목적과 모델 성능을 비교하여 선택해야 한다.

### 실습 34. 같은 단어 사전을 사용해서 Count와 TF-IDF를 비교하려면

기본 설정이 같더라도 두 Vectorizer를 별도로 fit()하면 일반적으로 비슷한 단어 사전을 만들 수 있습니다.

하지만 정확한 비교를 위해서는 각각의 feature 이름이 어떻게 만들어졌는지 직접 확인하는 것이 안전합니다.

In [26]:
# CountVectorizer가 만든 단어 집합
count_feature_set = set(
    count_vectorizer.get_feature_names_out()
)

# TfidfVectorizer가 만든 단어 집합
tfidf_feature_set = set(
    tfidf_vectorizer.get_feature_names_out()
)

# 단어 수와 일치 여부 확인
print("Count 단어 수:", len(count_feature_set))
print("TF-IDF 단어 수:", len(tfidf_feature_set))
print("같은 단어 집합인가?:", count_feature_set == tfidf_feature_set)

Count 단어 수: 2192
TF-IDF 단어 수: 2192
같은 단어 집합인가?: True


### 실행 결과

CountVectorizer와 TfidfVectorizer가 생성한 단어 수는 모두 **2,192개**이며, 단어 집합도 동일한 것으로 확인되었다.

같은 제목과 기본 토큰 설정을 사용했기 때문에 단어 사전은 같지만, Count는 등장 횟수를 저장하고 TF-IDF는 가중치를 저장한다.

### 실습 35. stop_words 옵션 이해하기

Vectorizer에서도 불용어를 지정할 수 있습니다.

In [27]:
# 분석에서 제외할 불용어 지정
stop_words = [
    "그리고",
    "대한",
    "위한",
]

# 불용어가 적용된 TF-IDF Vectorizer 생성
vectorizer_with_stopwords = TfidfVectorizer(
    stop_words=stop_words,
)

# 실제 도서 제목에 적용
X_stopwords = vectorizer_with_stopwords.fit_transform(titles)

# 생성된 단어 목록 확인
stopword_terms = vectorizer_with_stopwords.get_feature_names_out()

print("적용 후 단어 수:", len(stopword_terms))
print("'그리고' 포함 여부:", "그리고" in stopword_terms)
print("'대한' 포함 여부:", "대한" in stopword_terms)
print("'위한' 포함 여부:", "위한" in stopword_terms)

적용 후 단어 수: 2190
'그리고' 포함 여부: False
'대한' 포함 여부: False
'위한' 포함 여부: False


### 실행 결과

불용어 적용 후 단어 수는 기존 2,192개에서 **2,190개**로 줄었다.

지정한 `그리고`, `대한`, `위한`은 모두 최종 단어 목록에 포함되지 않았다. 단어 수가 2개만 감소한 것으로 보아 세 단어 중 하나는 기존 단어 사전에도 없었던 것으로 판단된다.

불용어는 분석 결과를 확인한 뒤 의미가 적은 단어만 선택하여 적용해야 한다.

### 실습 36. max_features 옵션 이해하기

단어 수가 너무 많을 때 최대 feature 수를 제한할 수 있습니다.

In [28]:
# 사용할 단어 수를 최대 1,000개로 제한
limited_vectorizer = TfidfVectorizer(
    max_features=1000,
)

# 실제 도서 제목에 적용
X_limited = limited_vectorizer.fit_transform(titles)

# 제한 전후 단어 수 확인
print("제한 전 단어 수:", len(tfidf_terms))
print(
    "제한 후 단어 수:",
    len(limited_vectorizer.get_feature_names_out()),
)
print("제한 후 행렬 크기:", X_limited.shape)

제한 전 단어 수: 2192
제한 후 단어 수: 1000
제한 후 행렬 크기: (989, 1000)


### 실행 결과

`max_features=1000`을 적용한 결과 단어 수가 **2,192개에서 1,000개로 제한**되었다.

문서 수는 989개로 유지되어 행렬 크기는 `(989, 1000)`으로 변경되었다. 단어 수 제한은 분석 목적과 모델 성능을 비교한 후 결정해야 한다.

### 실습 37. min_df 옵션 이해하기

min_df는 너무 적은 문서에서만 등장하는 단어를 제외할 때 사용할 수 있습니다.

예를 들어 다음과 같습니다.

In [29]:
# 2개 이상의 문서에 등장한 단어만 사용
vectorizer_min_df = TfidfVectorizer(
    min_df=2,
)

# 실제 도서 제목에 적용
X_min_df = vectorizer_min_df.fit_transform(titles)

# 적용 전후 단어 수 확인
print("적용 전 단어 수:", len(tfidf_terms))
print(
    "min_df 적용 후 단어 수:",
    len(vectorizer_min_df.get_feature_names_out()),
)
print("적용 후 행렬 크기:", X_min_df.shape)

적용 전 단어 수: 2192
min_df 적용 후 단어 수: 619
적용 후 행렬 크기: (989, 619)


### 실행 결과

`min_df=2`를 적용한 결과 단어 수가 **2,192개에서 619개로 감소**했다.

한 문서에만 등장한 단어가 제외되어 행렬 크기는 `(989, 619)`로 변경되었다. 드문 단어도 도서의 특징을 나타낼 수 있으므로 제외 전후의 분석 결과를 비교해야 한다.

### 실습 38. max_df 옵션 이해하기

max_df는 지나치게 많은 문서에 등장하는 단어를 자동으로 제외할 때 사용할 수 있습니다.

In [30]:
# 전체 문서의 95%를 초과하여 등장하는 단어 제외
vectorizer_max_df = TfidfVectorizer(
    max_df=0.95,
)

# 실제 도서 제목에 적용
X_max_df = vectorizer_max_df.fit_transform(titles)

# 적용 전후 단어 수 확인
print("적용 전 단어 수:", len(tfidf_terms))
print(
    "max_df 적용 후 단어 수:",
    len(vectorizer_max_df.get_feature_names_out()),
)
print("적용 후 행렬 크기:", X_max_df.shape)

적용 전 단어 수: 2192
max_df 적용 후 단어 수: 2192
적용 후 행렬 크기: (989, 2192)


### 실행 결과

`max_df=0.95`를 적용했지만 단어 수는 **2,192개로 변하지 않았고**, 행렬 크기도 `(989, 2192)`로 동일했다.

전체 문서의 95%를 초과하여 등장한 단어가 없기 때문에 제외된 단어가 없는 것으로 확인되었다.

### 실습 39. ngram_range 개념 맛보기

기본 Vectorizer는 보통 한 단어씩 feature를 만듭니다.

이를 unigram이라고 생각할 수 있습니다.

In [31]:
# 한 단어와 두 단어 조합을 함께 사용
bigram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
)

# 작은 예제 문장에 적용
X_bigram = bigram_vectorizer.fit_transform(sample_docs)

# 생성된 단어와 단어 조합 확인
bigram_terms = bigram_vectorizer.get_feature_names_out()

print("생성된 특징:")
print(bigram_terms)

print("행렬 크기:", X_bigram.shape)

생성된 특징:
['데이터' '데이터 분석' '머신러닝' '분석' '분석 입문' '입문' '파이썬' '파이썬 데이터' '파이썬 머신러닝']
행렬 크기: (3, 9)


### 실행 결과

`ngram_range=(1, 2)`를 적용한 결과 한 단어와 연속된 두 단어 조합을 포함하여 총 **9개의 특징**이 생성되었다.

`데이터`, `분석`, `파이썬`과 같은 Unigram뿐 아니라 `데이터 분석`, `파이썬 데이터`, `파이썬 머신러닝`과 같은 Bigram도 포함되었다. 이에 따라 행렬 크기는 `(3, 9)`로 나타났다.

### 실습 40. Chapter 02 형태소 분석 결과와 연결하기

Chapter 02에서는 Kiwi를 이용해 명사 중심으로 단어를 추출했습니다.

같은 전처리 철학을 Vectorizer에도 적용할 수 있습니다.

예를 들어 먼저 각 제목을 형태소 분석해서 사용할 단어만 문자열로 다시 연결할 수 있습니다.

In [32]:
# Kiwi와 TfidfVectorizer 불러오기
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer

# Kiwi 형태소 분석기 생성
kiwi = Kiwi()

# 사용할 품사와 제외할 불용어 설정
USE_TAGS = {"NNG", "NNP", "SL"}
STOP_WORDS = {"도서", "책"}

# 제목에서 사용할 단어만 추출하는 함수
def extract_terms(text):
    terms = []

    for token in kiwi.tokenize(str(text)):
        word = token.form.strip()

        # 명사, 고유명사, 영문이 아니면 제외
        if token.tag not in USE_TAGS:
            continue

        # 한 글자 단어 제외
        if len(word) < 2:
            continue

        # 불용어 제외
        if word in STOP_WORDS:
            continue

        terms.append(word)

    return terms


# 추출한 단어를 다시 공백으로 연결
processed_titles = titles.apply(
    lambda text: " ".join(extract_terms(text))
)

# 전처리된 제목 앞의 5개 확인
print("전처리 결과:")
display(processed_titles.head())

# Kiwi 전처리 결과에 TF-IDF 적용
kiwi_tfidf_vectorizer = TfidfVectorizer()
X_kiwi_tfidf = kiwi_tfidf_vectorizer.fit_transform(
    processed_titles
)

# 기본 방식과 Kiwi 방식 비교
print("기본 TF-IDF 행렬:", X_tfidf.shape)
print("Kiwi TF-IDF 행렬:", X_kiwi_tfidf.shape)

전처리 결과:


0          세네카 오늘
1    홍정기 장수 근육 혁명
2            싯다르타
3          머니 트렌드
4              남매
Name: 상품명, dtype: str

기본 TF-IDF 행렬: (989, 2192)
Kiwi TF-IDF 행렬: (989, 1378)


### 실행 결과

Kiwi 형태소 분석을 적용하여 명사·고유명사·영문만 추출하고, 한 글자 단어와 불용어를 제외했다.

기본 TF-IDF 행렬은 `(989, 2192)`였지만 Kiwi 전처리 후에는 `(989, 1378)`로 단어 수가 감소했다. 이는 품사, 글자 수, 불용어 조건에 맞지 않는 단어가 제외되었기 때문이다.

Kiwi 방식이 항상 더 좋은 것은 아니므로 이후 모델 성능을 비교하여 선택해야 한다.

### 실습 41. Custom tokenizer 방식은 언제 사용할까?

Vectorizer에 직접 tokenizer 함수를 전달할 수도 있습니다.

예를 들어 다음과 같은 형태입니다.

In [33]:
# Vectorizer에서 사용할 Kiwi tokenizer 함수
def kiwi_tokenizer(text):
    return extract_terms(text)


# Kiwi tokenizer를 직접 사용하는 TF-IDF Vectorizer 생성
custom_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_tokenizer,
    token_pattern=None,  # 기본 토큰 기준 사용하지 않음
    lowercase=False,     # 영문을 자동으로 소문자로 바꾸지 않음
)

# 원본 제목에 바로 적용
X_custom_tfidf = custom_vectorizer.fit_transform(titles)

# 결과 확인
print("전처리 문자열 방식:", X_kiwi_tfidf.shape)
print("Custom tokenizer 방식:", X_custom_tfidf.shape)
print(
    "생성된 단어 수:",
    len(custom_vectorizer.get_feature_names_out()),
)

전처리 문자열 방식: (989, 1378)
Custom tokenizer 방식: (989, 1357)
생성된 단어 수: 1357


### 실행 결과

전처리 문자열 방식의 행렬 크기는 `(989, 1378)`, Custom tokenizer 방식은 `(989, 1357)`로 나타났다.

두 방식 모두 같은 Kiwi 추출 함수를 사용했지만, 전처리 문자열 방식은 이후 Vectorizer의 기본 토큰 기준이 다시 적용되고 Custom tokenizer 방식은 Kiwi가 반환한 단어를 직접 사용하므로 단어 수에 차이가 발생할 수 있다.

Custom tokenizer 방식은 간결하지만 중간 전처리 결과를 확인하기 어려우므로, 학습 단계에서는 전처리 문자열을 먼저 확인하는 방식이 더 이해하기 쉽다.

### 실습 42. Vectorizer가 만든 결과를 시각적으로 이해하기

작은 예제에서 다음 표를 떠올립니다.

### Vectorizer 행렬 구조 이해

Vectorizer가 만든 행렬에서 각 행은 하나의 문서이고, 각 열은 하나의 단어이다.

| 문서 | 데이터 | 머신러닝 | 분석 | 입문 | 파이썬 |
| --- | ---: | ---: | ---: | ---: | ---: |
| 파이썬 데이터 분석 | 1 | 0 | 1 | 0 | 1 |
| 파이썬 머신러닝 | 0 | 1 | 0 | 0 | 1 |
| 데이터 분석 입문 | 1 | 0 | 1 | 1 | 0 |

CountVectorizer에서는 셀 값이 단어의 등장 횟수를 의미한다. TF-IDF에서도 행과 열의 구조는 같지만, 셀 값은 단순한 횟수가 아니라 TF-IDF 가중치이다.

즉, 하나의 문장은 여러 단어의 숫자값으로 이루어진 하나의 벡터로 표현된다.

### 실습 43. 행렬에서 0이 많다는 의미

각 도서 제목은 짧습니다.

전체 데이터에는 수많은 단어가 있지만 한 제목에는 그중 일부만 등장합니다.

따라서 행렬은 다음처럼 0이 많아집니다.

### 행렬에 0이 많은 이유

전체 데이터에는 2,192개의 단어가 있지만 하나의 도서 제목에는 그중 일부 단어만 등장한다.

따라서 각 제목의 벡터에는 등장한 단어의 위치에만 값이 들어가고, 나머지 대부분은 0으로 표시된다. 이렇게 0이 많은 행렬을 희소 행렬이라고 한다.

희소 행렬은 0이 아닌 값만 효율적으로 저장하며, 이후 코사인 유사도에서도 이러한 TF-IDF 벡터를 사용한다.

### 실습 44. 0이 아닌 값의 개수 확인하기

전체 행렬의 셀 수와 실제 저장된 0이 아닌 값의 개수를 비교할 수 있습니다.

In [34]:
# TF-IDF 행렬의 행과 열 개수
rows, cols = X_tfidf.shape

# 전체 셀 개수
total_cells = rows * cols

# 실제로 저장된 0이 아닌 값의 개수
non_zero_cells = X_tfidf.nnz

# 0인 셀의 비율 계산
zero_ratio = 1 - (non_zero_cells / total_cells)

# 결과 확인
print("전체 셀 수:", total_cells)
print("0이 아닌 셀 수:", non_zero_cells)
print("0의 비율:", round(zero_ratio, 4))

전체 셀 수: 2167888
0이 아닌 셀 수: 3921
0의 비율: 0.9982


### 실행 결과

TF-IDF 행렬의 전체 셀은 **2,167,888개**이며, 이 중 0이 아닌 셀은 **3,921개**이다.

0의 비율은 **0.9982(약 99.82%)**로, 대부분의 값이 0인 희소 행렬임을 확인했다. 따라서 전체를 일반 배열로 변환하기보다 희소 행렬 형태로 저장하는 것이 효율적이다.

### 실습 45. AI에게 결과 해석을 요청할 때 주의하기

### TF-IDF 결과 해석 시 주의사항

이번 분석의 문서 수는 **989개**, 단어 수는 **2,192개**이다.

평균 TF-IDF 상위 단어는 `2027`, `2026`, `해커스`, `기본서`, `세트`, `에디션`, `기출문제집`, `토익`, `the`, `해커스경찰` 순으로 나타났다.

이 단어들은 현재 도서 제목 데이터에서 상대적으로 두드러진 텍스트 특징이다. TF-IDF가 높다는 이유만으로 도서 판매의 원인이나 독자의 선호라고 단정할 수 없다.

AI에게 해석을 요청할 때는 실제 결과를 함께 제공하고, 결과에 없는 숫자를 임의로 만들지 않도록 해야 한다.

### 실습 46. Count와 TF-IDF 차이를 Markdown으로 정리하기

### CountVectorizer와 TF-IDF 비교

CountVectorizer는 각 도서 제목에서 단어가 등장한 횟수를 숫자로 표현한다.

TF-IDF는 한 제목에서의 단어 빈도뿐 아니라 전체 제목에서 해당 단어가 얼마나 흔한지도 함께 반영한다.

실제 결과에서 상위 9개 단어는 같았지만 이후 순위에는 차이가 나타났다. 따라서 단순히 자주 등장하는 단어와 각 문서의 특징을 상대적으로 잘 나타내는 단어는 같은 개념이 아님을 확인했다.

이번 결과는 현재 도서 제목 데이터와 Vectorizer 설정을 기준으로 한 텍스트 특징이며, 판매 원인이나 독자 선호로 직접 해석하지 않았다.

### 실습 47. 일부 도서 결과를 직접 검증하기

무작위로 몇 개 제목을 선택해 TF-IDF 주요 단어를 확인해 봅니다.

In [35]:
# 검증할 도서의 행 번호
sample_indices = [0, 10, 20]

# 각 도서의 제목과 주요 TF-IDF 단어 확인
for index in sample_indices:
    # 데이터 범위를 벗어나지 않는 경우에만 실행
    if index < len(titles):
        print("=" * 60)
        display(show_top_tfidf_terms(index, top_n=5))

도서 제목: 세네카, 오늘을 빼앗기고 있는 당신에게


,단어,TF-IDF
0,세네카,0.458262
1,오늘을,0.458262
2,빼앗기고,0.458262
3,당신에게,0.458262
4,있는,0.399979


도서 제목: 군주론 : 쇼츠 에디션


,단어,TF-IDF
0,쇼츠,0.655469
1,군주론,0.618580
2,에디션,0.433265


도서 제목: 트렌드 코리아 2027


,단어,TF-IDF
0,코리아,0.690265
1,트렌드,0.638404
2,2027,0.340550


### 검증 결과

선택한 세 도서의 TF-IDF 상위 단어가 모두 실제 제목에 존재하는 것을 확인했다.

`세네카`, `쇼츠`, `코리아`처럼 각 제목을 구분하는 단어가 상대적으로 높은 값을 보였다. 반면 여러 제목에 등장하는 `있는`, `에디션`, `2027`은 상대적으로 낮은 값을 보였다.

숫자나 기호가 비정상적인 특징으로 추출된 경우는 없었으며, 결과가 원본 제목과 일치했다.

### 실습 48. 결과가 이상할 때 확인할 순서

TF-IDF 결과가 예상과 다르더라도 바로 코드를 전부 바꾸지 않습니다.

### TF-IDF 결과가 이상할 때 확인할 순서

1. 원본 제목이 정상적인지 확인한다.
2. 결측치와 빈 문자열 처리 여부를 확인한다.
3. Vectorizer가 만든 단어 목록을 확인한다.
4. 한 글자 제외 등 기본 토큰화 기준을 확인한다.
5. 불용어 설정이 과도하지 않은지 확인한다.
6. `min_df`, `max_df`, `max_features` 옵션을 확인한다.
7. Kiwi 형태소 분석 적용 여부를 확인한다.
8. 해당 문서 행의 0이 아닌 단어와 값을 원본 제목과 비교한다.

결과가 예상과 다르더라도 코드를 바로 변경하지 않고, 원본 데이터와 전처리 조건부터 순서대로 확인해야 한다.

### 실습 49. 학습용 행렬 저장하기

희소 행렬을 저장하려면 scipy 기능을 사용할 수 있습니다.

In [36]:
# 희소 행렬 저장 기능과 경로 확인 기능 불러오기
from scipy.sparse import save_npz
from pathlib import Path

# Count 행렬 저장
save_npz(
    "chapter03_count_matrix.npz",
    X_count,
)

# TF-IDF 행렬 저장
save_npz(
    "chapter03_tfidf_matrix.npz",
    X_tfidf,
)

# 파일 저장 여부 확인
print(
    "Count 행렬 저장:",
    Path("chapter03_count_matrix.npz").exists(),
)
print(
    "TF-IDF 행렬 저장:",
    Path("chapter03_tfidf_matrix.npz").exists(),
)

Count 행렬 저장: True
TF-IDF 행렬 저장: True


### 실행 결과

Count 행렬을 `chapter03_count_matrix.npz`, TF-IDF 행렬을 `chapter03_tfidf_matrix.npz`로 저장했다.

두 파일의 존재 여부가 모두 `True`로 확인되어 희소 행렬이 정상적으로 저장되었다. 해당 파일은 이번 Chapter의 개념 학습과 결과 확인용으로만 사용한다.

### 실습 50. 왜 Chapter 04에서는 전체 데이터에 먼저 fit하면 안 될까?

이번 Chapter에서는 Vectorizer 자체를 이해하기 위해 전체 제목을 사용했습니다.

### 전체 데이터에 먼저 fit하면 안 되는 이유

이번 Chapter에서는 Vectorizer의 원리를 학습하기 위해 전체 도서 제목에 `fit_transform()`을 사용했다.

하지만 모델 성능을 평가할 때 전체 데이터에 먼저 Vectorizer를 `fit`하면 단어 사전과 IDF를 계산하는 과정에서 테스트 데이터의 정보까지 사용하게 된다. 이는 모델이 학습 단계에서 보지 않아야 할 정보를 미리 알게 되는 데이터 누수이다.

따라서 분류 모델을 만들 때는 train/test를 먼저 나누고, Vectorizer는 train 데이터에만 `fit`해야 한다.

### 실습 51. Chapter 04에서 사용할 올바른 순서 미리 보기

### Chapter 04의 올바른 처리 순서

분류 모델을 만들 때는 다음 순서로 진행한다.

1. 원본 데이터에서 입력값 `X`와 정답 `y`를 준비한다.
2. 데이터를 train과 test로 먼저 나눈다.
3. Vectorizer를 train 데이터에만 `fit`한다.
4. train 데이터에는 `fit_transform()`을 사용한다.
5. test 데이터에는 학습된 기준으로 `transform()`만 사용한다.
6. 변환된 train 데이터로 모델을 학습한다.
7. 변환된 test 데이터로 예측하고 성능을 평가한다.

```python
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### 실습 52. fit과 transform을 구분해서 설명해 보기

이번 Chapter를 마치기 전에 다음 문장을 본인의 말로 설명해 봅니다.

### fit과 transform의 차이

- `fit`: 데이터에서 사용할 단어 사전과 IDF 같은 기준을 학습한다.
- `transform`: 이미 학습한 기준을 사용하여 텍스트를 숫자로 변환한다.
- `fit_transform`: 기준 학습과 숫자 변환을 한 번에 수행한다.

모델 평가에서는 train 데이터에 `fit_transform()`을 사용하고, test 데이터에는 `transform()`만 사용해야 한다.

중요한 것은 **어떤 데이터에 fit했는가**이며, 테스트 데이터의 정보가 학습 과정에 포함되지 않도록 해야 한다.

### 실습 53. Vectorizer 옵션을 기록하기

결과 재현을 위해 어떤 설정으로 Vectorizer를 만들었는지 Notebook에 기록합니다.

기본 설정을 사용했다면 다음처럼 작성할 수 있습니다.

### Chapter 03 벡터화 조건

- 데이터: `book_bestseller_clean.csv`
- 분석 컬럼: `상품명`
- 사용한 제목 수: 989개
- CountVectorizer: 기본 설정
- TfidfVectorizer: 기본 설정
- 기본 단어 수: 2,192개
- 실습 목적: 전체 문서에서 벡터화 구조와 단어 특징 확인
- 추가 실험: `stop_words`, `max_features`, `min_df`, `max_df`, `ngram_range`, Kiwi 전처리
- 모델 평가용 Vectorizer: Chapter 04에서 train 데이터 기준으로 별도 학습 예정

이번 Chapter의 전체 데이터 `fit_transform()`은 개념 학습과 탐색을 위한 것이다. 실제 모델 평가에서는 train/test를 먼저 나누고 train 데이터에만 Vectorizer를 `fit`한다.

### 실습 54. 재현 가능한 코드 구조로 정리하기

Notebook 마지막에는 핵심 코드를 지나치게 흩어놓지 말고 한 번 정리해 봅니다.

In [37]:
# 필요한 라이브러리 불러오기
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
)

# 데이터 파일 경로
DATA_PATH = "book_bestseller_clean.csv"

# 1. 데이터 불러오기
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

# 2. 상품명 문자열 정리
titles = df_books["상품명"].fillna("").astype(str).str.strip()
titles = titles[titles != ""].reset_index(drop=True)

# 3. CountVectorizer 적용
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(titles)
count_terms = count_vectorizer.get_feature_names_out()

# 4. 단어별 전체 등장 횟수 계산
count_sums = np.asarray(X_count.sum(axis=0)).ravel()

count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums,
}).sort_values(
    "전체등장횟수",
    ascending=False,
).reset_index(drop=True)

# 5. TfidfVectorizer 적용
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(titles)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

# 6. 단어별 평균 TF-IDF 계산
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "평균_TFIDF": mean_tfidf,
}).sort_values(
    "평균_TFIDF",
    ascending=False,
).reset_index(drop=True)

# 7. 최종 결과 확인
print("사용한 제목 수:", len(titles))
print("Count shape:", X_count.shape)
print("TF-IDF shape:", X_tfidf.shape)

print("\nCount 상위 30개")
display(count_summary.head(30))

print("\n평균 TF-IDF 상위 30개")
display(tfidf_summary.head(30))

사용한 제목 수: 989
Count shape: (989, 2192)
TF-IDF shape: (989, 2192)

Count 상위 30개


,단어,전체등장횟수
0,2027,93
1,2026,57
2,해커스,38
3,기본서,30
4,세트,25
5,에디션,22
6,기출문제집,21
7,토익,20
8,the,18
9,기념,17



평균 TF-IDF 상위 30개


,단어,평균_TFIDF
0,2027,0.022807
1,2026,0.014421
2,해커스,0.010282
3,기본서,0.009226
4,세트,0.008907
5,에디션,0.007372
6,기출문제집,0.007274
7,토익,0.006968
8,the,0.006315
9,해커스경찰,0.005791


### 실행 결과

전체 핵심 코드를 다시 실행한 결과 **989개의 제목**에서 Count와 TF-IDF 행렬이 모두 `(989, 2192)`로 생성되었다.

Count 상위 단어는 `2027`, `2026`, `해커스`, `기본서`, `세트` 순이었으며, 평균 TF-IDF 상위 단어도 초반에는 같은 순서를 보였다.

전체 코드를 한 셀에서 재실행해도 이전 실습과 동일한 결과가 나와 분석 과정이 정상적으로 재현되는 것을 확인했다.

### 실습 55. Notebook 최종 실행 확인

제출 또는 다음 Chapter로 넘어가기 전에 Notebook을 처음부터 다시 실행합니다.

In [38]:
# 최종 결과 파일과 주요 변수 확인
from pathlib import Path

print("사용한 제목 수:", len(titles))
print("Count 행렬:", X_count.shape)
print("TF-IDF 행렬:", X_tfidf.shape)

print(
    "Count 상위 단어 CSV:",
    Path("chapter03_count_top_terms.csv").exists(),
)
print(
    "TF-IDF 상위 단어 CSV:",
    Path("chapter03_tfidf_top_terms.csv").exists(),
)
print(
    "Count 희소 행렬:",
    Path("chapter03_count_matrix.npz").exists(),
)
print(
    "TF-IDF 희소 행렬:",
    Path("chapter03_tfidf_matrix.npz").exists(),
)

사용한 제목 수: 989
Count 행렬: (989, 2192)
TF-IDF 행렬: (989, 2192)
Count 상위 단어 CSV: True
TF-IDF 상위 단어 CSV: True
Count 희소 행렬: True
TF-IDF 희소 행렬: True


### 최종 실행 확인

Notebook 전체 셀을 다시 실행한 결과 오류 없이 완료되었다.

Count와 TF-IDF 행렬은 모두 `(989, 2192)`로 동일하게 재현되었다. 상위 단어 CSV 파일과 희소 행렬 파일도 모두 `True`로 확인되어 정상적으로 저장되었다.

### 실습 56. 이번 Chapter에서 꼭 기억할 개념

### Chapter 03 핵심 정리

1. 머신러닝이 텍스트를 계산할 수 있도록 숫자 벡터로 변환해야 한다.
2. Bag of Words는 단어 순서보다 단어의 등장 여부와 횟수에 초점을 둔다.
3. CountVectorizer는 행을 문서, 열을 단어, 값을 등장 횟수로 표현한다.
4. TF-IDF는 현재 문서의 단어 빈도와 전체 문서에서의 희소성을 함께 반영한다.
5. 높은 TF-IDF는 현실에서 절대적으로 중요하다는 뜻이 아니라, 현재 문서 집합에서 상대적으로 두드러진다는 의미이다.
6. `fit`은 단어 사전과 기준을 학습하고, `transform`은 학습된 기준으로 텍스트를 숫자로 변환한다.
7. 모델 평가에서는 데이터 누수를 막기 위해 train 데이터에만 Vectorizer를 `fit`해야 한다.

### 실습 57. Chapter 03 결과물 정리

### Chapter 03 결과물

#### 생성 파일

- `chapter03.ipynb`
- `chapter03_count_top_terms.csv`
- `chapter03_tfidf_top_terms.csv`
- `chapter03_count_matrix.npz`
- `chapter03_tfidf_matrix.npz`

#### 최종 결과

- 사용한 도서 제목: 989개
- 생성된 단어: 2,192개
- Count 행렬 크기: `(989, 2192)`
- TF-IDF 행렬 크기: `(989, 2192)`
- 0이 아닌 셀: 3,921개
- 0의 비율: 약 99.82%

CountVectorizer를 통해 단어 등장 횟수를 숫자로 표현하고, TfidfVectorizer를 통해 전체 문서에서의 흔함을 반영한 가중치를 계산했다.

Count와 평균 TF-IDF의 상위 단어는 일부 순위 차이를 보였으며, 단순히 자주 등장하는 단어와 각 문서의 특징을 상대적으로 잘 나타내는 단어가 같은 개념은 아님을 확인했다.

이번 결과는 벡터화 원리를 학습하기 위한 탐색 결과이다. Chapter 04의 모델 평가에서는 train/test를 먼저 분리하고 train 데이터에만 Vectorizer를 `fit`해야 한다.